# Qwen3-8B: re-evaluate saved labels

Evaluates the teacher, baseline, and fine-tuned labels against the original tickets using the team's reference-free rubric (faithfulness, specificity, coherence). No Qwen model loading, training, or Git operations. Use a **CPU runtime**.

Run cells in order. The judge cell sends ticket text and candidate labels to the configured Anthropic judge and incurs API charges: **up to 60 successful judgments**, plus retries if needed. Completed judgments are checkpointed after every call. Original experiment files are read only.

This evaluates **4 test clusters × 5 prompts per model**. It does not establish a reliable winner or general performance. This run was already inspected during hyperparameter selection, so it is not a fresh test set.


In [ ]:
# 1. Mount Drive and locate the original run.
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
EXPERIMENT = Path('/content/drive/MyDrive/slm-distillation/experiments/20260922T214033Z_qwen3-8b-e5-lr1e4_a32b189c')
RUN = EXPERIMENT / 'runs/20260922_2140_Qwen3-8B_ep5'
INPUT_DIR = RUN / 'evaluation'
DATA_FILE = RUN / 'labeled_data.csv'
CONFIG_FILE = EXPERIMENT / 'phase1_config.yaml'
assert INPUT_DIR.is_dir() and DATA_FILE.is_file() and CONFIG_FILE.is_file(), 'Check the paths before continuing.'
print('Original run found:', RUN.name)


In [ ]:
# 2. Install only the small config reader. No model dependencies.
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'pyyaml'], check=True)


In [ ]:
# 3. Validate inputs and construct the blinded judge requests (no API calls).
import csv, json, hashlib, random, re, time, urllib.request, urllib.error
import yaml
cfg = yaml.safe_load(CONFIG_FILE.read_text())
TOP_K = int(cfg['top_k'])
JUDGE_MODEL = cfg['evaluation']['judge_llm']['model']
assert cfg['evaluation']['judge_llm']['provider'] == 'anthropic'
DIMS = ('faithfulness', 'specificity', 'coherence')
TAGS = ('teacher', 'baseline', 'finetuned')
RUBRIC = 'You are evaluating the quality of a label generated for a cluster of IT/HR/CX support tickets.\n\nBelow are the top support tickets from the cluster and the candidate label to evaluate.\n\nRate the CANDIDATE LABEL on these three dimensions, each on a scale of 1–5:\n1. Faithfulness (1–5): Does the label accurately reflect the dominant theme of these tickets?\n2. Specificity (1–5): Is the label specific enough to distinguish this cluster from others?\n3. Coherence (1–5): Is the label clearly written, well-formed, and professional?\n\nScoring guide:  5 = excellent  |  4 = good  |  3 = acceptable  |  2 = poor  |  1 = wrong\n\n--- TICKETS ---\n{tickets}\n\n--- CANDIDATE LABEL ---\n{candidate}\n\nRespond ONLY with valid JSON (no markdown fences, no extra text):\n{{"faithfulness": <int>, "specificity": <int>, "coherence": <int>, "reasoning": "<one sentence>"}}\n'
# Rubric source: qwen2.5-3b/code/phase1/evaluation/llm_judge.py
# Content blob: 08ba2b2093cdbece249c5d698b25bb2fd947e61f
# Model tags and reference labels are NOT sent to the judge.
def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()
def read_jsonl(path):
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
def key(row):
    return (int(row['cluster_id']), row['prompt_id'])
preds = {tag: read_jsonl(INPUT_DIR / f'{tag}_predictions.jsonl') for tag in TAGS}
keysets = [{key(r) for r in preds[tag]} for tag in TAGS]
assert keysets[0] == keysets[1] == keysets[2]
assert all(len(preds[tag]) == len(keysets[i]) == 20 for i, tag in enumerate(TAGS)), 'Unexpected or duplicate prediction rows.'
clusters = sorted({cid for cid, pid in keysets[0]})
assert len(clusters) == 4
assert keysets[0] == {(cid, f'P{i}') for cid in clusters for i in range(1, 6)}
with DATA_FILE.open(newline='') as f:
    data = list(csv.DictReader(f))
tickets = {}
for cid in clusters:
    group = [r for r in data if int(r['cluster_id']) == cid and float(r['ticket_rank_in_cluster']) <= TOP_K]
    group.sort(key=lambda r: float(r['ticket_rank_in_cluster']))
    assert group and all(r['ticket_details'].strip() for r in group)
    ranks = [float(r['ticket_rank_in_cluster']) for r in group]
    assert len(ranks) == len(set(ranks)), 'Duplicate ticket ranks.'
    tickets[cid] = [r['ticket_details'] for r in group]
for r in preds['teacher']:
    column = 'cluster_name_claude_haiku_4_5_' + r['prompt_id']
    values = {x[column] for x in data if int(x['cluster_id']) == int(r['cluster_id'])}
    assert values == {r['generated_label']}, 'Teacher labels and dataset do not match.'
jobs = []
for tag in TAGS:
    for row in preds[tag]:
        assert isinstance(row['generated_label'], str) and row['generated_label'].strip()
        cid, pid = key(row)
        block = '\n'.join(f'{i}. {text}' for i, text in enumerate(tickets[cid], 1))
        rendered = RUBRIC.format(tickets=block, candidate=row['generated_label'])
        job = dict(model=tag, cluster_id=cid, prompt_id=pid, generated_label=row['generated_label'], prompt=rendered)
        job['job_id'] = digest([tag, cid, pid, rendered, JUDGE_MODEL, 0.0])
        jobs.append(job)
random.Random(42).shuffle(jobs)
# Fingerprint isolates this evaluation from any different input/rubric/model.
fingerprint = digest({'jobs': jobs, 'judge_model': JUDGE_MODEL, 'temperature': 0.0, 'max_tokens': 300})
OUT = RUN / ('reference_free_' + fingerprint[:12])
OUT.mkdir(exist_ok=True)
SCORES = OUT / 'scores.jsonl'
manifest = dict(fingerprint=fingerprint, judge_model=JUDGE_MODEL, temperature=0.0, top_k=TOP_K,
                clusters=clusters, count=len(jobs), rubric=RUBRIC,
                input_sha256={p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in
                              [DATA_FILE, CONFIG_FILE] + [INPUT_DIR / f'{t}_predictions.jsonl' for t in TAGS]})
(OUT / 'manifest.json').write_text(json.dumps(manifest, indent=2))
print(f'Validated: {len(jobs)} labels, {len(clusters)} independent clusters, top_k={TOP_K}.')
print('Judge:', JUDGE_MODEL, '| Outputs:', OUT)


### Run the judge

Add `ANTHROPIC_API_KEY` in Colab Secrets and enable notebook access. Run the next cell to make the judge calls. The API receives each candidate separately without its model identity or teacher reference. The rubric matches the team code; formatting and API settings are recorded for reproducibility. Judge sampling may still vary at temperature zero.

If interrupted, rerun this notebook; it resumes completed judgments. Malformed or out-of-range scores stop the run instead of being counted as zeros.


In [ ]:
# 4. Paid judge calls; no GPU required.
from google.colab import userdata
api_key = userdata.get('ANTHROPIC_API_KEY')
assert api_key, 'Enable ANTHROPIC_API_KEY in Colab Secrets.'
def score_job(job):
    body = dict(model=JUDGE_MODEL, max_tokens=300, temperature=0.0,
                messages=[{'role': 'user', 'content': job['prompt']}])
    for attempt in range(5):
        req = urllib.request.Request('https://api.anthropic.com/v1/messages',
            data=json.dumps(body).encode(), method='POST',
            headers={'x-api-key': api_key, 'anthropic-version': '2023-06-01', 'content-type': 'application/json'})
        try:
            with urllib.request.urlopen(req, timeout=90) as response:
                payload = json.load(response)
                request_id = response.headers.get('request-id')
            break
        except urllib.error.HTTPError as exc:
            status = exc.code
            if status not in (429, 500, 502, 503, 504, 529) or attempt == 4:
                raise RuntimeError(f'Judge HTTP {status}; completed calls remain saved. Check model access, credits, and API settings.') from None
            time.sleep(min(2 ** (attempt + 1), 30))
    assert payload.get('stop_reason') != 'max_tokens', 'Judge response truncated; no score saved.'
    raw = ''.join(c.get('text', '') for c in payload.get('content', []) if c.get('type') == 'text').strip()
    raw = re.sub(r'^```(?:json)?\s*|\s*```$', '', raw).strip()
    result = json.loads(raw)
    assert all(type(result.get(d)) is int and 1 <= result[d] <= 5 for d in DIMS), 'Invalid scores; no zero substitution.'
    assert isinstance(result.get('reasoning'), str)
    return {d: result[d] for d in DIMS} | dict(reasoning=result['reasoning'],
        composite=sum(result[d] for d in DIMS)/3, usage=payload.get('usage', {}),
        response_model=payload.get('model'), request_id=request_id)
existing = read_jsonl(SCORES) if SCORES.exists() else []
assert len({r['job_id'] for r in existing}) == len(existing)
expected_ids = {j['job_id'] for j in jobs}
assert {r['job_id'] for r in existing} <= expected_ids
assert all(all(type(r.get(d)) is int and 1 <= r[d] <= 5 for d in DIMS) for r in existing)
done = {r['job_id'] for r in existing}
print('Already saved:', len(done), '| Remaining:', len(jobs) - len(done))
for job in jobs:
    if job['job_id'] in done:
        continue
    result = {k:v for k,v in job.items() if k != 'prompt'} | score_job(job)
    with SCORES.open('a') as f:
        f.write(json.dumps(result, ensure_ascii=False) + '\n')
        f.flush()
    done.add(job['job_id'])
    print(f'{len(done)}/{len(jobs)} saved')
    time.sleep(1)
print('Complete. Original experiment results were not changed.')


In [ ]:
# 5. Summaries and paired comparisons at the CLUSTER level.
from statistics import mean
rows = read_jsonl(SCORES)
assert len(rows) == len(jobs) == len({r['job_id'] for r in rows}), 'Finish all judge calls before summarizing.'
assert {r['job_id'] for r in rows} == {j['job_id'] for j in jobs}
def write_csv(path, values):
    with path.open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(values[0]))
        writer.writeheader()
        writer.writerows(values)
summary = []
cluster_means = {}
for tag in TAGS:
    subset = [r for r in rows if r['model'] == tag]
    summary.append(dict(model=tag, predictions=len(subset), clusters=len(clusters),
                        **{d: mean(r[d] for r in subset) for d in (*DIMS, 'composite')}))
    for cid in clusters:
        cluster_means[tag, cid] = mean(r['composite'] for r in subset if r['cluster_id'] == cid)
paired = [dict(cluster_id=cid, teacher=cluster_means['teacher',cid],
               baseline=cluster_means['baseline',cid], finetuned=cluster_means['finetuned',cid],
               ft_minus_baseline=cluster_means['finetuned',cid]-cluster_means['baseline',cid],
               ft_minus_teacher=cluster_means['finetuned',cid]-cluster_means['teacher',cid]) for cid in clusters]
write_csv(OUT / 'summary.csv', summary)
write_csv(OUT / 'paired_by_cluster.csv', paired)
write_csv(OUT / 'judge_scores.csv', [{k:v for k,v in r.items() if k != 'usage'} for r in rows])
for r in summary:
    print(r['model'], '| composite:', round(r['composite'],3), '|', {d: round(r[d],3) for d in DIMS})
print('Mean paired FT minus baseline:', round(mean(r['ft_minus_baseline'] for r in paired),3))
print('Mean paired FT minus teacher:', round(mean(r['ft_minus_teacher'] for r in paired),3))
print('Only 4 independent clusters. Treat differences as descriptive; do not claim significance or equivalence.')
print('Download summary.csv and paired_by_cluster.csv from:', OUT)


### After the run

Upload `summary.csv` and `paired_by_cluster.csv` here. Retain `scores.jsonl` and `manifest.json` for reproducibility. This notebook does not measure inference latency and its scores use a different rubric from the original 4.20/5 result.

No GitHub pull is required: this notebook is self-contained. After checking it runs in Colab, commit it as `notebooks/03_qwen3_reference_free.ipynb` on your `btt-setup-ak` branch. Keep API keys in Colab Secrets. This notebook has not made any API calls during preparation.
